# NNUE Training Pipeline

End-to-end notebook: data generation → Stockfish labeling → training → `.nbai` export.

**Runtime:** GPU (T4 or better). Runtime → Change runtime type → T4 GPU.

**Persistence:** Checkpoints and datasets are saved to Google Drive so training can resume after session timeout.

---
## ⚙️ 1 — Setup

In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Configure these paths once ───────────────────────────────────────────────
GITHUB_TOKEN = ''          # Personal access token with repo:read scope
REPO_URL     = 'https://github.com/YOUR_ORG/NowChessSystems.git'  # replace
DRIVE_ROOT   = '/content/drive/MyDrive/NowChess'
REPO_DIR     = f'{DRIVE_ROOT}/NowChessSystems'
PYTHON_DIR   = f'{REPO_DIR}/modules/official-bots/python'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    !git clone --depth=1 "{auth_url}" "{REPO_DIR}"
    print('Repo cloned to Drive.')
else:
    !git -C "{REPO_DIR}" pull --ff-only
    print('Repo updated.')

In [ ]:
# Install Python dependencies
!pip install -q chess tqdm rich zstandard

# Stockfish for position labeling
!apt-get install -q -y stockfish
import shutil
STOCKFISH_PATH = shutil.which('stockfish') or '/usr/games/stockfish'
print(f'Stockfish: {STOCKFISH_PATH}')

# Add pipeline source to path
import sys
sys.path.insert(0, f'{PYTHON_DIR}/src')
sys.path.insert(0, PYTHON_DIR)
print('Python path configured.')

---
## 🗄️ 2 — Data

Choose **one** of the two options below:
- **Option A** — generate FEN positions with random play, then label them with Stockfish.
- **Option B** — upload an existing `labeled.jsonl` from your machine or Drive.

In [ ]:
from pathlib import Path

# Paths (all on Drive so they survive session restarts)
DATA_DIR      = Path(DRIVE_ROOT) / 'training_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
POSITIONS_FILE = DATA_DIR / 'positions.txt'   # raw FENs
LABELED_FILE   = DATA_DIR / 'labeled.jsonl'   # FEN + eval pairs

print(f'Data directory: {DATA_DIR}')

In [ ]:
# ── Option A: Generate + label ────────────────────────────────────────────────
# Adjust NUM_POSITIONS to taste. 50 000 trains in ~10 min on T4;
# 200 000+ gives better generalisation.
NUM_POSITIONS    = 50_000
STOCKFISH_DEPTH  = 12
LABEL_WORKERS    = 4       # parallel Stockfish processes
MIN_MOVE         = 5       # skip opening book moves
MAX_MOVE         = 60

from generate import play_random_game_and_collect_positions
from label    import label_positions_with_stockfish

print(f'Generating {NUM_POSITIONS:,} positions...')
count = play_random_game_and_collect_positions(
    str(POSITIONS_FILE),
    total_positions=NUM_POSITIONS,
    samples_per_game=1,
    min_move=MIN_MOVE,
    max_move=MAX_MOVE,
    num_workers=4,
)
print(f'{count:,} positions written to {POSITIONS_FILE}')

print('Labeling with Stockfish (this is the slow step)...')
ok = label_positions_with_stockfish(
    str(POSITIONS_FILE),
    str(LABELED_FILE),
    STOCKFISH_PATH,
    depth=STOCKFISH_DEPTH,
    num_workers=LABEL_WORKERS,
)
if ok:
    print(f'Labeled dataset saved: {LABELED_FILE}')
else:
    print('ERROR: labeling failed')

In [ ]:
# ── Option B: Upload existing labeled.jsonl ───────────────────────────────────
# Run this cell instead of Option A if you already have a labeled dataset.
#
# To upload from local machine:
#   from google.colab import files
#   uploaded = files.upload()   # pick your labeled.jsonl
#   import shutil, os
#   shutil.move(next(iter(uploaded)), str(LABELED_FILE))
#
# Or copy from Drive:
#   import shutil
#   shutil.copy('/content/drive/MyDrive/path/to/labeled.jsonl', str(LABELED_FILE))

import os
if LABELED_FILE.exists():
    lines = sum(1 for _ in open(LABELED_FILE))
    print(f'Ready: {lines:,} labeled positions at {LABELED_FILE}')
else:
    print('No labeled.jsonl found — run Option A first or upload one.')

---
## 🏋️ 3 — Train

Standard training runs a fixed number of epochs.  
**Burst mode** is better for Colab: it repeatedly restarts from the best checkpoint within a time budget, surviving session disconnects gracefully.

In [ ]:
from train import train_nnue, burst_train, DEFAULT_HIDDEN_SIZES

WEIGHTS_DIR = Path(DRIVE_ROOT) / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = str(WEIGHTS_DIR / 'nnue_weights.pt')

# ── Training hyperparameters ──────────────────────────────────────────────────
HIDDEN_SIZES      = DEFAULT_HIDDEN_SIZES   # [1536, 1024, 512, 256]
BATCH_SIZE        = 16384
EPOCHS            = 100
EARLY_STOPPING    = 10                     # None to disable
SUBSAMPLE_RATIO   = 1.0

# Resume from latest checkpoint if one exists
checkpoints = sorted(WEIGHTS_DIR.glob('nnue_weights_v*.pt'))
CHECKPOINT = str(checkpoints[-1]) if checkpoints else None
if CHECKPOINT:
    print(f'Resuming from checkpoint: {CHECKPOINT}')
else:
    print('Starting training from scratch.')

In [ ]:
# ── Standard training ─────────────────────────────────────────────────────────
# Use this when you have a reliable long-running session.

train_nnue(
    data_file=str(LABELED_FILE),
    output_file=OUTPUT_FILE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    checkpoint=CHECKPOINT,
    use_versioning=True,
    early_stopping_patience=EARLY_STOPPING,
    subsample_ratio=SUBSAMPLE_RATIO,
    hidden_sizes=HIDDEN_SIZES,
)

In [ ]:
# ── Burst training (recommended for Colab free tier) ─────────────────────────
# Restarts from the global best each time early stopping fires.
# Set BURST_MINUTES to slightly less than the Colab session limit (~70 min).

BURST_MINUTES      = 70
EPOCHS_PER_SEASON  = 30
BURST_PATIENCE     = 8

burst_train(
    data_file=str(LABELED_FILE),
    output_file=OUTPUT_FILE,
    duration_minutes=BURST_MINUTES,
    epochs_per_season=EPOCHS_PER_SEASON,
    early_stopping_patience=BURST_PATIENCE,
    batch_size=BATCH_SIZE,
    initial_checkpoint=CHECKPOINT,
    use_versioning=True,
    subsample_ratio=SUBSAMPLE_RATIO,
    hidden_sizes=HIDDEN_SIZES,
)

---
## 📦 4 — Export

Convert the best `.pt` checkpoint to the `.nbai` binary format read by `NbaiLoader` in Scala.

In [ ]:
from export import export_to_nbai

NBAI_FILE = Path(DRIVE_ROOT) / 'nnue_weights.nbai'

# Pick the latest versioned checkpoint
checkpoints = sorted(WEIGHTS_DIR.glob('nnue_weights_v*.pt'))
if not checkpoints:
    raise FileNotFoundError('No checkpoints found in ' + str(WEIGHTS_DIR))

latest = checkpoints[-1]
print(f'Exporting {latest.name} → {NBAI_FILE.name}')

export_to_nbai(
    weights_file=str(latest),
    output_file=str(NBAI_FILE),
    trained_by='colab',
)
print('Export complete.')

---
## ⬇️ 5 — Download

Download the `.nbai` weights file and the latest `.pt` checkpoint to your local machine.

Place `nnue_weights.nbai` in `modules/official-bots/src/main/resources/` and rebuild the native image.

In [ ]:
from google.colab import files

if NBAI_FILE.exists():
    files.download(str(NBAI_FILE))
    print(f'Downloading {NBAI_FILE.name}')
else:
    print('No .nbai file found — run the Export cell first.')

checkpoints = sorted(WEIGHTS_DIR.glob('nnue_weights_v*.pt'))
if checkpoints:
    latest = checkpoints[-1]
    files.download(str(latest))
    print(f'Downloading checkpoint {latest.name}')
else:
    print('No .pt checkpoint found.')